# Notebook 5: Implied vs Realised Volatility
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import norm
from scipy.optimize import brentq
import sys; sys.path.insert(0,'../src')
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#0d1117',
    'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9',
    'xtick.color':'#8b949e','ytick.color':'#8b949e',
    'text.color':'#c9d1d9','grid.color':'#21262d',
    'axes.titlecolor':'#f0f6fc','legend.facecolor':'#161b22',
    'legend.edgecolor':'#30363d','font.size':11
})
COLORS = ['#58a6ff','#3fb950','#f78166','#d2a8ff','#ffa657','#79c0ff']
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
spy_ret = returns['SPY']
S0, r, sigma = 100, 0.05, 0.20
print(f'Loaded {len(spy_ret):,} observations')


Loaded 2,609 observations


In [2]:
from implied_vol import vol_risk_premium
rv_63=spy_ret.rolling(63).std()*np.sqrt(252)*100
iv_est=rv_63*1.08+pd.Series(np.random.normal(0,1.5,len(rv_63)),index=spy_ret.index)
vrp_res=vol_risk_premium(rv_63.dropna(),iv_est.dropna())
print(f"Mean VRP      : {vrp_res["mean_vrp"]}%")
print(f"Std VRP       : {vrp_res["std_vrp"]}%")
print(f"% Positive VRP: {vrp_res["pct_positive"]}%")

Mean VRP      : 1.4065%
Std VRP       : 1.5312%
% Positive VRP: 82.3%


In [3]:
vrp=vrp_res["series"]
fig,axes=plt.subplots(2,1,figsize=(14,9),facecolor="#0d1117")
axes[0].plot(rv_63.index,rv_63,color="#3fb950",linewidth=1.2,label="Realised Vol 63d",alpha=0.9)
axes[0].plot(iv_est.index,iv_est,color="#58a6ff",linewidth=1.2,label="Implied Vol (est.)",alpha=0.9)
axes[0].fill_between(iv_est.index,rv_63,iv_est,where=iv_est>=rv_63,alpha=0.15,color="#58a6ff",label="Vol Risk Premium")
axes[0].fill_between(iv_est.index,rv_63,iv_est,where=iv_est<rv_63,alpha=0.15,color="#f78166",label="IV < RV")
axes[0].set_title("Implied vs Realised Volatility",color="#f0f6fc")
axes[0].set_ylabel("Volatility (%)",color="#8b949e"); axes[0].legend(); axes[0].grid(True,alpha=0.3)
axes[1].fill_between(vrp.index,vrp,0,where=vrp>=0,color="#3fb950",alpha=0.6,label="Positive VRP")
axes[1].fill_between(vrp.index,vrp,0,where=vrp<0,color="#f78166",alpha=0.6,label="Negative VRP")
axes[1].axhline(vrp.mean(),color="#58a6ff",linewidth=1.5,linestyle="--",label=f"Mean={vrp.mean():.1f}%")
axes[1].axhline(0,color="#8b949e",linewidth=0.8)
axes[1].set_title("Volatility Risk Premium",color="#f0f6fc")
axes[1].set_ylabel("VRP (%)",color="#8b949e"); axes[1].legend(); axes[1].grid(True,alpha=0.3)
for ax in axes: ax.tick_params(colors="#8b949e")
plt.tight_layout()
plt.savefig("../results/07_implied_vs_realised.png",dpi=150,bbox_inches="tight",facecolor="#0d1117")
plt.show()
print("Chart saved.")

Chart saved.
